##### Cell 1 — Imports and shared data

In [1]:

# ==========================================================
# IT3091 - Class Imbalance Verification
# Responsibility: IT24101808
# ==========================================================

import sys
from pathlib import Path

import pandas as pd
import numpy as np

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data_preparation import (
    load_and_clean_data,
    separate_features_target
)

from src.validation import create_train_test_split

from src.imbalance import (
    get_sampler,
    build_model_pipeline
)

df = load_and_clean_data(project_root)
X, y = separate_features_target(df)

X_train, X_test, y_train, y_test = (
    create_train_test_split(X, y)
)

print("Training class counts:")
print(y_train.value_counts().sort_index())

print("\nTesting class counts:")
print(y_test.value_counts().sort_index())


Training class counts:
Churn
0    4139
1    1495
Name: count, dtype: int64

Testing class counts:
Churn
0    1035
1     374
Name: count, dtype: int64


##### Cell 2 — Check available strategies

In [2]:

# Responsibility: IT24101808

no_sampler = get_sampler("none")

oversampler = get_sampler("random_oversampling")

print("No balancing:", no_sampler)
print("Random oversampling:", oversampler)


No balancing: None
Random oversampling: RandomOverSampler(random_state=42)


##### Cell 3 — Inspect random oversampling on training data

In [3]:

# Responsibility: IT24101808

sampler = get_sampler("random_oversampling")

X_sampled, y_sampled = sampler.fit_resample(
    X_train,
    y_train
)

print("Before oversampling:")
print(y_train.value_counts().sort_index())

print("\nAfter oversampling:")
print(pd.Series(y_sampled).value_counts().sort_index())

print("\nOriginal training records:", len(X_train))
print("Resampled training records:", len(X_sampled))

print("\nTest records remain:", len(X_test))


Before oversampling:
Churn
0    4139
1    1495
Name: count, dtype: int64

After oversampling:
Churn
0    4139
1    4139
Name: count, dtype: int64

Original training records: 5634
Resampled training records: 8278

Test records remain: 1409


##### Cell 4 — Verify pipeline construction

In [4]:

# Responsibility: IT24101808 - Integration

from sklearn.linear_model import LogisticRegression

example_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

baseline_pipeline = build_model_pipeline(
    model=example_model,
    scale_numeric=True,
    add_engineered_features=False,
    imbalance_strategy="none"
)

oversampling_pipeline = build_model_pipeline(
    model=LogisticRegression(
        max_iter=1000,
        random_state=42
    ),
    scale_numeric=True,
    add_engineered_features=True,
    imbalance_strategy="random_oversampling"
)

print("Baseline pipeline:")
print(list(baseline_pipeline.named_steps.keys()))

print("\nOversampling pipeline:")
print(list(oversampling_pipeline.named_steps.keys()))


Baseline pipeline:
['features', 'preprocessor', 'model']

Oversampling pipeline:
['features', 'preprocessor', 'sampler', 'model']
